In [ ]:
# Install and Import Packages.
import sys
from Bio import Entrez as ez
import pandas as pd
from dotenv import load_dotenv
import os
from time import sleep

In [ ]:

ez_functions = [func for func in dir(ez) if callable(getattr(ez, func)) and not func.startswith('_')]

# Loop through each function name in your list
for func_name in ez_functions:
    # Get the actual function object from the 'ez' module
    function_object = getattr(ez, func_name)
    
    # Get the docstring, handle cases where it might be empty
    docstring = function_object.__doc__
    if docstring:
        # Split the docstring into lines and take the first non-empty one
        first_line = docstring.strip().split('\n')[0]
        print(f"{func_name}: {first_line}")
    else:
        print(f"{func_name}: No description available.")

In [ ]:
# Load environment variables from .env file
load_dotenv()

ez.email = os.environ.get('EMAIL')
# ez.api_key = os.environ.get('API_KEY')
SEARCH_QUERY = os.environ.get('SEARCH_QUERY')
DATABASE = os.environ.get('DATABASE')
RETTYPE = os.environ.get('RETTYPE')
RETMODE = os.environ.get('RETMODE')

print(SEARCH_QUERY)
print(ez.email)


In [ ]:

print(SEARCH_QUERY)

print("Running esearch...")
handle = ez.esearch(db=DATABASE,
                        term=SEARCH_QUERY,
                        usehistory="y") # IMPORTANT: Use the history server

search_results = ez.read(handle)
handle.close()

# Get the total count and the history server identifiers
count = int(search_results["Count"])
webenv = search_results["WebEnv"]
query_key = search_results["QueryKey"]

print(f"Found {count} results.")

In [ ]:
batch_size = 100
all_records = []

print("Fetching records in batches...")
for start in range(0, count, batch_size):
    end = min(count, start + batch_size)
    print(f"Fetching records from {start+1} to {end}")
    
    fetch_handle = ez.efetch(db = DATABASE,
                                 rettype = RETTYPE, # Request XML format for full data
                                 retmode= RETMODE,
                                 retstart=start,
                                 retmax=batch_size,
                                 webenv=webenv, # Use the history server identifiers
                                 query_key=query_key)
    
    # Biopython's ez.read can parse the XML into a structured Python object
    records = ez.read(fetch_handle)
    fetch_handle.close()
    
    # The actual articles are in the 'PubmedArticle' list
    all_records.extend(records['PubmedArticle'])
    
    # Be nice to the server! A short delay between requests.
    sleep(0.3)

print("Finished fetching all records.")

In [ ]:
import pandas as pd
# Assuming 'all_records' is a list of dictionaries you've already loaded.
# all_records = ... 

parsed_articles = []

for article in all_records:
    citation = article['MedlineCitation']
    
    # FIX: Define article_info here to access the nested 'Article' dictionary
    article_info = citation.get('Article', {})

    # Use article_info to access all subsequent fields
    title = article_info.get('ArticleTitle', 'No Title Found')

    abstract_parts = article_info.get('Abstract', {}).get('AbstractText', [])
    abstract = ' '.join(abstract_parts) if abstract_parts else 'No Abstract Found'

    journal_info = article_info.get('Journal', {})
    journal_name = journal_info.get('Title', 'No Journal Found')
    
    pub_date = journal_info.get('JournalIssue', {}).get('PubDate', {})
    year = pub_date.get('Year', pub_date.get('MedlineDate', 'No Year Found'))

    pmid = citation.get('PMID', '')

    # Now this line will work correctly because article_info is defined
    author_list = article_info.get('AuthorList', [])
    authors = []
    for author_data in author_list:
        fore_name = author_data.get('ForeName', '')
        last_name = author_data.get('LastName', '')
        if fore_name and last_name:
            authors.append(f"{fore_name} {last_name}")
    
    authors_str = '; '.join(authors) if authors else 'No Authors Found'

    parsed_articles.append({
        'PMID': str(pmid),
        'Title': title,
        'Authors': authors_str,
        'Abstract': abstract,
        'Journal': journal_name,
        'Year': year
    })

df = pd.DataFrame(parsed_articles)

# Optional: Reorder columns for a more logical layout in the CSV
if not df.empty:
    df = df[['PMID', 'Title', 'Abstract', 'Year', 'Authors', 'Journal']]
    
df.to_csv("systematic_review_results.csv", index=False)

print("Successfully parsed and saved results to systematic_review_results.csv")
print(df.head())